# Challenge 3: Can We Trust What the Random Forest Tells Us?

Customer behaviour, feature engineering, model evidence, and decisions

CS-4120 | Wednesday, October 28, 2026 | Team challenge | 75 minutes

## Your mission

A retailer wants to contact customers who might stop shopping. An analyst recommends a Random Forest and says the most important feature reveals why customers leave.

Your team is the review committee. Decide what information is safe to use, engineer one meaningful customer-behaviour feature, examine the analyst's evidence, and recommend what the retailer should do next.

There is no model to train and almost no code to write. Your work is to think, question, listen, and defend decisions using evidence.

This is a fictional classroom case. Customer examples and model results are illustrative, not results from an actual retailer or a real trained model.

### What you need

- Your group of four.
- TraCE, your AI thinking partner. Ask for explanations and counterarguments, not a finished assignment.
- This notebook as a case packet. You may run the optional self-check at the end.
- One shared Word document, submitted by one group member. A Word answer template is provided. No notebook or ZIP submission is needed.

### Class plan

| Time | Team task |
|---|---|
| 0-6 min | Roles and individual positions |
| 6-20 min | Audit the proposed features |
| 20-35 min | Design and defend an engineered feature |
| 35-49 min | Evaluate model evidence and Random Forest importance |
| 49-61 min | Challenge your reasoning with TraCE |
| 61-69 min | Decide what to do for actual customers |
| 69-75 min | Report to a neighbouring team and finish your memo |

## 1. Build your team and make an initial prediction | 0-6 min

Assign a different starting responsibility to each person. In a group of three, combine any two responsibilities.

| Role | Responsibility |
|---|---|
| Behaviour detective | Ask what customer behaviour the data really represents |
| Feature designer | Explain how the proposed features could be constructed |
| Evidence skeptic | Ask what could make the model results misleading |
| TraCE facilitator / recorder | Make sure everyone speaks; record the team's reasoning, not an AI transcript |

Each member gets 2 mins, without TraCE, to answer: "What could make a customer look likely to leave even if they are not really leaving?"

Before proceeding, agree on one example you expect could fool a prediction model. Keep that example in mind when you see the evidence.

Team rule: Every member must propose or critique at least one feature before you consult TraCE. Disagreement is encouraged if it is explained respectfully.

## Case card: Harbour Goods

Harbour Goods is a fictional retailer selling hobby supplies online and in stores. Some customers shop weekly, some only during promotions, and some place one large order during the holiday season.

Every Monday at 9 a.m., the retailer could score customers who bought at least once during the preceding year.

- Proposed target: `1` if the customer makes no purchase in the following 60 days; `0` otherwise.
- Question: Who appears at risk of not purchasing in the next 60 days?
- Proposed action: Offer selected customers a discount to encourage another purchase.
- Operational limit: Contact at most 500 out of 3,000 eligible customers in a typical decision period.
- In the later-period validation cohort, 750 of those 3,000 customers have target `1` (25%).
- The historical data contains repeated snapshots of some customers. Promotions and shopping patterns change over time.

Notice two problems before choosing features:

1. Does no purchase for 60 days necessarily mean a customer has left?
2. If a model predicts a customer will not purchase, does that prove a discount will change the customer's behaviour?

Keep the prediction question separate from the intervention question. You do not have data measuring whether the discount works.

## 2. The feature audit | 6-20 min

The analyst provides these candidate columns. "Monday" means the prediction time. A feature is only safe if it could have been computed from information available at that time.

| Code | Candidate column | Definition supplied by analyst |
|---|---|---|
| F1 | `days_since_last_purchase` | Number of days from the last purchase to Monday |
| F2 | `orders_last_90d` | Purchases during the 90 days ending before Monday |
| F3 | `discount_share_last_90d` | Proportion of those purchases using a discount |
| F4 | `purchases_next_60d` | Number of purchases in the 60 days following Monday |
| F5 | `future_offer_opened` | Whether the customer opens the retention email sent after scoring |
| F6 | `lifetime_spend_export` | Lifetime spend in the latest data export, with no as-of date documented |
| F7 | `customer_id` | Permanent unique identifier for the customer |
| F8 | `account_age_asof` | Days since the customer's first registered purchase, calculated as of Monday |
| F9 | `month_of_year` | Calendar month on the scoring Monday |
| F10 | `returns_last_90d` | Product returns recorded before Monday, during the previous 90 days |

Discuss each feature as one of: usable, exclude as predictor, or verify before using. Some features can be known at prediction time but still be poor predictors or misleading explanations.

Your team must include in Q2 of the memo:

- Three features you would consider using, with reasons.
- Two you would exclude as predictors, with reasons.
- One you would verify before using, stating exactly what information you need.

Challenge the first person's suggestion: Is the feature about behaviour, the customer's history, a data collection process, or information from the future?

Do not ask TraCE to sort the whole table for you.

## 3. Engineer one feature that captures behaviour | 20-35 min

Choose exactly one feature-engineering idea to defend. Each member proposes an option first; then the group decides together. You may choose an idea below or propose another that could be computed before Monday.

| Technique | Possible feature | A complication worth discussing |
|---|---|---|
| Difference / trend | `orders_previous_30d - orders_31_to_60d` | What does the sign mean, and how does it behave for new customers? |
| Ratio | `discounted_orders_90d / total_orders_90d` | What happens when the denominator is zero? |
| Rolling aggregation | Number of distinct product categories purchased in 90 days | Can the system recover category history at each past Monday? |
| Time feature | Month or historical seasonal shopping pattern | Could seasonal customers be classified as churners unfairly? |
| Interaction | Combine recency and prior purchase frequency | Would a Random Forest already capture some of this interaction? |

State your design in ordinary language or a simple formula. Then explain:

1. What customer behaviour is this feature trying to represent?
2. Which exact historical records and time window are needed?
3. What edge case, missing value, or customer subgroup could make it misleading?
4. How would it enter a pipeline before a Random Forest? If you compare logistic regression, what preprocessing might differ?

Ask a teammate who preferred a different feature to make the strongest argument against your choice. Record the alternative the team considered. A feature that is plausible does not need to be perfect.

Conceptual pipeline to defend: select a historical snapshot -> construct allowed features using that snapshot cutoff -> create training and validation periods -> learn any fitted preprocessing using training folds only -> fit a simple baseline and Random Forest -> rank risk -> evaluate on future periods. The final test period stays untouched during selection.

## 4. Open the analyst's model evidence | 35-49 min

These are illustrative evaluation summaries supplied by the fictional analyst. Each validation cohort has 3,000 customers, with 750 actual non-purchasers. The retailer may flag its top 500 scores. These are not interchangeable evaluation cohorts: row-random and forward-time validation assess different deployment situations.

| Experiment | Features | Validation design | True non-purchasers among 500 flagged | Precision at 500 | Suitable evidence for next-period deployment? |
|---|---|---|---:|---:|---|
| A | Random Forest, including after-scoring fields | Row-random | 470 | 94% | No |
| B | Random Forest, after-scoring fields removed | Row-random | 395 | 79% | Not sufficient |
| C | Random Forest, after-scoring fields and ID removed | Forward-time | 300 | 60% | More relevant |
| D | Logistic regression, same valid feature set as C | Same forward-time cohort as C | 285 | 57% | More relevant |

For the forward-time design, all feature values are constructed as they would have been available at each historical Monday. A gap is left so the 60-day outcome labels have matured before the next evaluation period. The final test period is still closed.

Discuss and answer in Q3:

1. Which comparison is valid for evaluating the model family, C versus D or A versus C? Why?
2. Why might A look excellent while being useless at prediction time? Why may B be overly optimistic about the next period?
3. In C, what is recall among the 750 actual non-purchasers if the retailer flags 500? Show the calculation.
4. Is the 15-customer difference between C and D enough to declare Random Forest the winner? What further evidence would you ask for?
5. Describe a pipeline that could support a credible future-period test without using test information to choose features or models.

The question is not "Which percentage is highest?" It is "Which evidence actually answers the retailer's question?"

## 5. What does Random Forest feature importance really say?

The analyst also sends two feature-importance summaries.

### Preliminary Random Forest: impurity importance on its training data

This version included fields that should have been audited more carefully. Notice that the analyst also introduced column names not in the original candidate list. What documentation would you ask for? Notice that the analyst also introduced column names not in the original candidate list. What documentation would you ask for? Notice that the analyst also introduced column names not in the original candidate list. What documentation would you ask for?

| Feature | Impurity importance |
|---|---:|
| `account_status_at_export` (updated after scoring) | 0.42 |
| `customer_id` | 0.16 |
| `days_since_last_purchase` | 0.15 |
| `orders_last_90d` | 0.11 |
| `spend_last_90d` | 0.09 |
| `discount_share_last_90d` | 0.07 |

### Audited Random Forest: forward-time validation

The next table is for the version with after-scoring fields and the identifier excluded, on the same forward-time validation cohort used for Experiment C.

| Feature | Impurity importance | Drop in validation precision at 500 if shuffled alone (percentage points) |
|---|---:|---:|
| `days_since_last_purchase` | 0.30 | 5.2 |
| `orders_last_90d` | 0.22 | 0.8 |
| `spend_last_90d` | 0.18 | 0.6 |
| `discount_share_last_90d` | 0.12 | 2.6 |
| `purchase_trend_30d` | 0.10 | 1.9 |
| `account_age_asof` | 0.08 | 0.7 |

Additional clue: `orders_last_90d` and `spend_last_90d` are highly correlated in this case. Shuffling them together reduces validation precision at 500 by 3.8 percentage points, more than shuffling either alone.

These example numbers are deliberately simplified. In practice permutation measurements also vary with the held-out sample and with the particular metric.

In Q4, explain to another student:

- Why was the preliminary importance ranking a warning rather than proof of customer behaviour?
- What is the difference between training impurity importance and shuffling a feature on held-out data?
- Why might correlated features look individually less important when both are available?
- Does a feature's high importance prove it causes customers to leave, or that a discount targeted using it will work?

Pick one claim from the analyst that your team would refuse to put in a business report. Rewrite it as a defensible claim.

## 6. Make TraCE a critical teammate | 49-61 min

You already have a provisional plan. Now use TraCE for two meaningful exchanges, not just to generate answers. The TraCE facilitator should let a different team member ask or answer at each exchange.

### Exchange 1: Learn and teach back

Suggested prompt to adapt:

> We are reviewing a customer-retention model. Explain the difference between purchase recency, purchase frequency, and a purchase trend using one concrete customer example. Then ask us two questions that test whether our engineered feature captures the behaviour we intend. Do not pick a feature for us and do not write our assignment.

After the response, one team member must teach the idea back to the others in their own words. Check whether TraCE made assumptions unsupported by the case card.

### Exchange 2: Challenge the team's provisional decision

Suggested prompt to adapt:

> Here is our current feature choice, why we chose it, and the reason we trust or doubt Experiment C: [insert our own reasoning]. Be a skeptical teammate. Give us one realistic counterexample and one competing interpretation of the feature-importance evidence. Ask us to defend our decision before offering advice. Do not decide for us.

Respond to TraCE's challenge. You may revise your recommendation or keep it with a stronger justification. Both are legitimate if supported.

In Q5 of your memo, record only:

- One idea your group originally held.
- The most useful challenge or explanation TraCE offered.
- What the group accepted, rejected, or revised, and why.
- One contribution from a human teammate that changed the discussion.

Do not submit a chat transcript. An AI answer is not evidence by itself; check it against the case and the model tables.

## 7. From predictions to actual customers | 61-69 min

The analyst shows four illustrative customer profiles. Risk scores estimate non-purchase in the next 60 days, not the benefit of a discount.

| Customer profile | Risk score | Behaviour the model summary does not fully capture |
|---|---:|---|
| A: Seasonal shopper | 0.89 | Makes one large gift order around the holidays each year; last purchase was 175 days ago |
| B: Sudden change | 0.73 | Purchased almost every week for months, but no purchase in the last 28 days |
| C: Promotion-dependent | 0.78 | Five recent orders, all with discounts; almost never buys without a promotion |
| D: New customer | 0.69 | Only one order, 12 days ago; very little history to establish a trend |

You cannot contact everyone. Choose two profiles to investigate first and explain what you would want to know before sending an offer. At least one teammate should argue for a different profile.

Now write your group's decision in Q6:

- Deploy the scores immediately, run a limited pilot with safeguards, or pause for further validation? Explain.
- What do the scores tell us and what can they not tell us about the effect of a discount?
- Name one concrete next check, such as verifying feature timestamps, comparing seasonal subgroups, measuring performance over several future periods, or testing discounts against a no-offer control group.

A confident answer should acknowledge uncertainty, not erase it.

## Group report and hand-in

Use the supplied editable Word answer template if helpful. Use the supplied editable Word answer template if helpful. Use the supplied editable Word answer template if helpful. One member submits one Word file to Moodle, named `Challenge-3_TeamName_Answers.docx`. Include your team name, members, and student IDs. Use the headings below and write concise group responses. There is no minimum word count, no required code, and no TraCE transcript.

Q1. What does the target mean, and which kind of customer might it misclassify?

Q2. Give three plausible features, two to exclude, one to verify, and your engineered feature. Include its construction, one edge case, and an alternative proposed by a teammate.

Q3. Compare Experiments C and D, calculate recall for Experiment C, and explain the split and pipeline you trust for model selection. What further evidence would help?

Q4. Explain two problems with the importance tables, including the correlation clue, and rewrite one overconfident statement the analyst might make.

Q5. Record one substantial TraCE challenge, your response to it, and how a teammate contributed. State what changed or why your decision survived the challenge.

Q6. Choose two customer profiles to investigate, then recommend deploy, pilot, or pause. Give a reason and one concrete next check. Explain why risk is not the same as benefit from an offer.

### What matters in assessment (20 points)

- Understanding the decision and customer context: 4
- Sound feature audit and engineered-feature reasoning: 4
- Interpretation of model evidence and feature importance: 4
- Critical, documented TraCE and teammate dialogue: 4
- Defensible team recommendation and communication: 4

The quality of reasoning matters, not who writes the most, how much code runs, or whether your conclusion matches another team's. If you cannot resolve something, explain what evidence is missing.

During the final minutes, trade your recommendation with a nearby team. Each team gets a brief opportunity to ask one difficult but constructive question.